# Notebook 02 — Alignment Experiments
**Owner: Member 2**

Tune DTW parameters and compare DTW vs. naive frame-by-frame matching.
Visualize the warping path on a test pair.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
from pose_extraction import load_keypoints
from normalization import normalize
from alignment import dtw_align, warping_path_to_timestamps

In [ ]:
# TODO: update paths after extracting keypoints
bench_kp = load_keypoints('../data/phrase_01/keypoints/benchmark_kp.npy')
user_kp  = load_keypoints('../data/phrase_01/keypoints/learner_kp.npy')
print('Benchmark shape:', bench_kp.shape)
print('Learner shape:  ', user_kp.shape)

In [ ]:
bench_norm = normalize(bench_kp)
user_norm  = normalize(user_kp)

bench_al, user_al, path = dtw_align(bench_norm, user_norm)
print(f'Aligned length T\' = {len(path)} frames')

In [ ]:
# Plot the warping path
bench_idx = [p[0] for p in path]
user_idx  = [p[1] for p in path]

plt.figure(figsize=(7, 5))
plt.plot(bench_idx, user_idx, lw=1)
plt.plot([0, max(bench_idx)], [0, max(user_idx)], 'r--', lw=1, label='diagonal (perfect sync)')
plt.xlabel('Benchmark frame index')
plt.ylabel('Learner frame index')
plt.title('DTW Warping Path')
plt.legend()
plt.show()

In [ ]:
# Ablation: naive frame-by-frame matching (truncate to shorter length)
from scoring import compute_joint_errors, per_part_error_over_time

# DTW-aligned
dtw_errors = compute_joint_errors(bench_al, user_al)
dtw_part   = per_part_error_over_time(dtw_errors)

# Naive: just crop to same length
T_min = min(len(bench_norm), len(user_norm))
naive_errors = compute_joint_errors(bench_norm[:T_min], user_norm[:T_min])
naive_part   = per_part_error_over_time(naive_errors)

for part in dtw_part:
    dtw_mean   = dtw_part[part].mean()
    naive_mean = naive_part[part].mean()
    print(f'{part:12s}  DTW mean={dtw_mean:.3f}   Naive mean={naive_mean:.3f}')